# Error analysis

Top-5 most-confused class pairs of the best model (selected by **validation** macro-F1), with example test images.
Run from the repo root after `make train` and `make table`; needs `data/Galaxy10_DECals.h5`.

In [ ]:
import os
os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None
import h5py, matplotlib.pyplot as plt, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix
from galaxy10.data import CLASSES
from galaxy10.evaluate import load_probs

metrics = pd.concat([pd.read_csv(f).assign(name=f.parts[-3]) for f in Path("results").glob("*/seed*/metrics.csv")])
best = metrics.groupby("name").val_macro_f1.mean().idxmax()
run = Path("results") / best / "seed0"
print("best model:", best)

In [ ]:
y, probs = load_probs(run / "test_probs.csv")
ids = pd.read_csv(run / "test_probs.csv").image_id.to_numpy()
pred = probs.argmax(1)
cm = confusion_matrix(y, pred, labels=range(10))
sym = cm + cm.T
pairs = sorted(((sym[i, j], i, j) for i in range(10) for j in range(i + 1, 10)), reverse=True)[:5]
pd.DataFrame([(CLASSES[i], CLASSES[j], cm[i, j], cm[j, i], s) for s, i, j in pairs],
             columns=["class A", "class B", "A predicted as B", "B predicted as A", "total"])

In [ ]:
images = h5py.File("data/Galaxy10_DECals.h5", "r")["images"]
fig, axes = plt.subplots(5, 6, figsize=(12, 10.5))
for row, (_, i, j) in zip(axes, pairs):
    a_as_b = ids[(y == i) & (pred == j)][:3]
    b_as_a = ids[(y == j) & (pred == i)][:3]
    for ax, k, (t, p) in zip(row, list(a_as_b) + list(b_as_a), [(i, j)] * len(a_as_b) + [(j, i)] * len(b_as_a)):
        ax.imshow(images[k])
        ax.set_title(f"true {t} → pred {p}", fontsize=8)
    for ax in row:
        ax.axis("off")
fig.suptitle("Most-confused pairs (test, seed 0)")
fig.tight_layout()

Disturbed (0) vs Merging (1) is expected to be among the hardest pairs: both labels come from overlapping volunteer answers about irregular morphology.